# PHOENIX Protected Split Demonstration

This notebook demonstrates the Week 2 protected dataset splitting workflow.

It **imports and reuses the implementation from**:

- `prepare_splits.py`
- `split_integrity.py`

The notebook does not duplicate the splitting or integrity-check logic. Its purpose is to provide clear demonstration evidence of the workflow and results.

> **Current limitation:** the development dataset does not contain the approved Sprint 2 phishing target, so target-based stratification and class-distribution checks remain disabled for now.


In [1]:
from pathlib import Path
import importlib
import pandas as pd

import prepare_splits as ps
import split_integrity as si

# Reload modules so notebook runs use the latest saved versions.
importlib.reload(ps)
importlib.reload(si)

print("Imported prepare_splits.py and split_integrity.py successfully.")


Imported prepare_splits.py and split_integrity.py successfully.


## 1. Load the source dataset

The source dataset is loaded using the function already defined in `prepare_splits.py`.


In [2]:
df = ps.load_dataset(ps.DATASET_PATH)

print("\nDataset shape:", df.shape)
print("Columns:")
for column in df.columns:
    print(f"- {column}")


Loaded dataset: phoenix_combined_dataset_large.xlsx
Rows: 2000
Columns: 10

Dataset shape: (2000, 10)
Columns:
- url
- text
- timestamp
- hazard_type
- hazard_severity
- hazard_timestamp
- hazard_location
- hazard_status
- alert_level
- source


## 2. Review the protected grouping field

The current split policy groups by unique `text` values so repeated copies of the same content cannot be placed in different splits.


In [3]:
print(f"Configured group column: {ps.GROUP_COLUMN}")
print(f"Total rows: {len(df)}")
print(f"Unique {ps.GROUP_COLUMN} groups: {df[ps.GROUP_COLUMN].nunique()}")
print(f"Configured target column: {ps.TARGET_COLUMN}")


Configured group column: text
Total rows: 2000
Unique text groups: 250
Configured target column: None


## 3. Validate the split configuration

This uses the validation logic from `prepare_splits.py` before any split is created.


In [4]:
ps.validate_configuration(df)
print("Configuration validation: PASS")


Configuration validation: PASS


## 4. Create the protected train/validation/test split

The split is performed on unique text groups rather than individual rows.

No fitted preprocessing, vectorisation, encoding, or model selection is performed before this split.


In [5]:
train_df, val_df, test_df = ps.protected_split(df)

ps.check_group_overlap(
    train_df,
    val_df,
    test_df,
)

ps.print_summary(
    df,
    train_df,
    val_df,
    test_df,
)



Unique text groups: 250

Protected split checks
----------------------
Train/Validation group overlap: 0
Train/Test group overlap: 0
Validation/Test group overlap: 0
Group overlap check: PASS

Split summary
-------------
Original:    2000 rows |  250 unique text groups
Train:       1444 rows |  175 unique text groups
Validation:   295 rows |   37 unique text groups
Test:         261 rows |   38 unique text groups

Rows after splitting: 2000
Row count reconciliation: PASS


## 5. Save the protected split files

The generated files are saved to the output directory configured in `prepare_splits.py`.


In [6]:
ps.save_splits(
    train_df,
    val_df,
    test_df,
)

print("\nGenerated files:")
for path in [
    ps.OUTPUT_DIR / "train.csv",
    ps.OUTPUT_DIR / "validation.csv",
    ps.OUTPUT_DIR / "test.csv",
]:
    print(f"- {path} | exists={path.exists()}")



Saved protected datasets
------------------------
Train:      split_data\train.csv
Validation: split_data\validation.csv
Test:       split_data\test.csv

Generated files:
- split_data\train.csv | exists=True
- split_data\validation.csv | exists=True
- split_data\test.csv | exists=True


## 6. Reload the generated files independently

The integrity script reloads the saved CSV files rather than checking only the in-memory DataFrames. This verifies the actual generated artefacts.


In [7]:
integrity_train = si.load_split(si.TRAIN_PATH, "Train")
integrity_val = si.load_split(si.VAL_PATH, "Validation")
integrity_test = si.load_split(si.TEST_PATH, "Test")


Train     :  1444 rows | 10 columns
Validation:   295 rows | 10 columns
Test      :   261 rows | 10 columns


## 7. Run split integrity checks

The checks below are imported directly from `split_integrity.py`.


In [8]:
results = {
    "Required columns": si.check_required_columns(
        integrity_train,
        integrity_val,
        integrity_test,
    ),
    "Schema consistency": si.check_schema_consistency(
        integrity_train,
        integrity_val,
        integrity_test,
    ),
    "Protected group overlap": si.check_group_overlap(
        integrity_train,
        integrity_val,
        integrity_test,
    ),
    "Row reconciliation": si.check_row_counts(
        integrity_train,
        integrity_val,
        integrity_test,
    ),
    "Missing group values": si.check_missing_group_values(
        integrity_train,
        integrity_val,
        integrity_test,
    ),
    "Exact duplicate review": si.check_exact_duplicates(
        integrity_train,
        integrity_val,
        integrity_test,
    ),
}

si.print_group_summary(
    integrity_train,
    integrity_val,
    integrity_test,
)

results["Target distribution"] = si.check_target_distribution(
    integrity_train,
    integrity_val,
    integrity_test,
)

si.print_final_result(results)



1. Required column checks
-------------------------
Train: PASS
Validation: PASS
Test: PASS

2. Schema consistency
---------------------
PASS - all splits contain 10 matching columns.

3. Protected group overlap
--------------------------
Train/Validation overlap: 0
Train/Test overlap:       0
Validation/Test overlap:  0
PASS - no protected groups cross split boundaries.

4. Row-count reconciliation
---------------------------
Train rows:      1444
Validation rows: 295
Test rows:       261
Combined rows:   2000
PASS - combined rows match expected total (2000).

5. Missing protected-group values
--------------------------------
Train     : 0 missing text values
Validation: 0 missing text values
Test      : 0 missing text values
PASS - no missing protected-group values.

6. Exact duplicate rows within splits
------------------------------------
Train     : 0 exact duplicate rows
Validation: 0 exact duplicate rows
Test      : 0 exact duplicate rows
PASS - no exact duplicate rows found.



## 8. Summary of results

This cell presents the main protected-split evidence in a concise table.


In [9]:
summary = pd.DataFrame(
    {
        "Partition": ["Train", "Validation", "Test"],
        "Rows": [
            len(integrity_train),
            len(integrity_val),
            len(integrity_test),
        ],
        "Unique text groups": [
            integrity_train[si.GROUP_COLUMN].nunique(),
            integrity_val[si.GROUP_COLUMN].nunique(),
            integrity_test[si.GROUP_COLUMN].nunique(),
        ],
    }
)

total_groups = summary["Unique text groups"].sum()
summary["Group share"] = summary["Unique text groups"] / total_groups

summary


,Partition,Rows,Unique text groups,Group share
0,Train,1444,175,0.700
1,Validation,295,37,0.148
2,Test,261,38,0.152


## 9. Current status

The current development workflow demonstrates that:

- the dataset can be loaded successfully;
- the split is protected by unique text group;
- no text group crosses train/validation/test boundaries;
- all source rows are accounted for;
- generated split files can be reloaded and independently validated;
- target-distribution checking remains intentionally skipped while no approved phishing target is configured.

When an approved phishing-labelled dataset becomes available, the same scripts can be reused with the approved target column and the protected splits regenerated.
